##Lab1: E-commerce Customer Behavior  

####Set up the file storage and constants.

In [0]:
CATALOG = "dbr_dev_ua5816bd"
SCHEMA = "elina_sharabura"
VOLUME = "raw_data"

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

ORDERS_FILE = "orders.csv"
CUSTOMERS_FILE = "customers.csv"

ORDERS_TABLE = "orders"
CUSTOMERS_TABLE = "customers"


spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

####Import raw CSV data into Databricks. 
Two CSV files are used for the core pipeline (orders.csv, customers.csv).

In [0]:
orders_df = spark.read.csv(
    f"{VOLUME_PATH}/{ORDERS_FILE}",
    header=True,
    inferSchema=True
)

customers_df = spark.read.csv(
    f"{VOLUME_PATH}/{CUSTOMERS_FILE}",
    header=True,
    inferSchema=True
)

Validate imported data (CSV schema and preview data (orders, customers))

In [0]:
orders_df.printSchema()
display(orders_df.limit(10))

customers_df.printSchema()
display(customers_df.limit(10))

**Import completed:** both DataFrames loaded successfully with correct types.

####Load the data into a Delta table using a Python notebook, saving it to the schema. 

In [0]:
orders_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.{ORDERS_TABLE}")

customers_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.{CUSTOMERS_TABLE}")

In [0]:
display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.{ORDERS_TABLE} LIMIT 10"))
display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.{CUSTOMERS_TABLE} LIMIT 10"))

**Delta tables created:** `orders` and `customers` are now persisted as managed Delta tables and are queryable via SQL.

####Practice basic Spark DataFrame operations (select, filter, join, groupBy). 

Select - pick only relevant columns from `orders`

In [0]:
orders_selected = orders_df.select("order_id", "customer_id", "order_time", "total_usd", "country")
display(orders_selected)

Filter - keep only high-value orders (total > $100, no discount)

In [0]:
high_value_orders = orders_df.filter(
    (orders_df.total_usd > 100) & (orders_df.discount_pct == 0)
)
display(high_value_orders)

Join - combine each order with customer details using a left join on `customer_id`, plus fix country column collision on join

In [0]:
customers_renamed = customers_df.withColumnRenamed("country", "customer_country")

orders_with_customers = orders_df.join(
    customers_renamed,
    on="customer_id",
    how="left"
)
display(orders_with_customers)

GroupBy - aggregate revenue and order count per country

In [0]:
from pyspark.sql.functions import sum as spark_sum, count

sales_by_country = orders_df.groupBy("country").agg(
    spark_sum("total_usd").alias("total_revenue"),
    count("order_id").alias("orders_count")
)
display(sales_by_country)

**DataFrame operations verified:** `select`, `filter`, `join` and `groupBy` successfully applied to `orders` and `customers`.

####Load data from an external API. 

Fetching current USD exchange rates from `api.frankfurter.app` (JSON response),
converting to a Spark DataFrame, and saving as a Delta table.

In [0]:
import requests

response = requests.get("https://api.frankfurter.app/latest?from=USD")
response.raise_for_status()   
data = response.json()

print(data)

In [0]:
from pyspark.sql import Row

api_rows = [
    Row(currency=currency, rate=float(rate), base=data["base"], rate_date=data["date"])
    for currency, rate in data["rates"].items()
]

exchange_rates_df = spark.createDataFrame(api_rows)
exchange_rates_df.printSchema()
display(exchange_rates_df)

In [0]:
EXCHANGE_RATES_TABLE = "exchange_rates"

exchange_rates_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.{EXCHANGE_RATES_TABLE}")

In [0]:
display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.{EXCHANGE_RATES_TABLE}"))

**API data loaded:** exchange rates fetched from `api.frankfurter.app`, converted from nested JSON to a flat DataFrame via `Row` objects (with explicit`float()` cast to avoid int float type conflicts across currencies), and saved as `dbr_dev_ua5816bd.elina_sharabura.exchange_rates` - verified queryable via SQL.

####Optional additions: explain Delta Lake benefits (ACID, time travel, schema enforcement). 


**ACID transactions** - every `write`/`saveAsTable` operation is atomic: it either fully succeeds or leaves the table in its previous, valid state. On a shared, auto-terminating cluster this matters a lot - if a write gets interrupted (cluster restart, network issue), the table won't end up half-written or corrupted.

**Time travel** - Delta keeps a transaction log (`_delta_log`) with the full history of changes to a table. This means I can query a previous version. This is useful if a bad `overwrite` or `merge` needs to be rolled back.

**Schema enforcement** - once a Delta table exists with a fixed schema, Delta rejects any write that doesn't match it (wrong column types, missing/extra columns), instead of silently corrupting the table. This protects tables from the kind of type inconsistency that can creep in from messy source data.